# Spatial Cost Inputs

Generates a per-location, per-technology cost overrides CSV for the global run.

The output CSV has columns: `lat, lon, tech, interest_rate, build_cost_multiplier,
land_cost_usd_per_km2_year, water_cost_usd_per_m3`.

**Current logic:**
- All locations use the YAML default `interest_rate`.
- Pure-ocean cells (`onshore_land_pct == 0`) get `build_cost_multiplier = 2.0` for all techs.
- Land and coastal cells get `build_cost_multiplier = 1.0`.

<!-- TODO: vary offshore costs by distance-to-coast and fixed/floating wind depth bands -->

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
import yaml


def find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for p in [start] + list(start.parents):
        if (p / 'model').is_dir():
            return p
    raise RuntimeError("Could not locate repo root containing 'model/'")


NOTEBOOK_DIR = Path('__file__').resolve().parent if '__file__' in globals() else Path.cwd()
repo_root = find_repo_root(NOTEBOOK_DIR)

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print('Repo root:', repo_root)

In [ ]:
# ----------------
# USER PARAMETERS
# ----------------

TECH_YAML = repo_root / 'inputs' / 'tech_config_ammonia_plant_2030_dea.yaml'
MAX_CAPACITIES_CSV = repo_root / 'data' / '20251222_max_capacities.csv'

# Output path (this file is consumed by the global run)
OUTPUT_CSV = repo_root / 'inputs' / 'spatial_cost_inputs.csv'

# Offshore build cost multiplier for pure-ocean cells
OFFSHORE_BUILD_COST_MULTIPLIER = 2.0

# Default water cost (applied uniformly; override per-location if needed)
DEFAULT_WATER_COST_USD_PER_M3 = 2.0

# Default land cost (only meaningful for cells with land)
DEFAULT_LAND_COST_USD_PER_KM2_YEAR = 0.0

print(f'Tech YAML: {TECH_YAML}')
print(f'Max capacities: {MAX_CAPACITIES_CSV}')
print(f'Output CSV: {OUTPUT_CSV}')
print(f'Offshore multiplier: {OFFSHORE_BUILD_COST_MULTIPLIER}x')

In [ ]:
# Load YAML tech config to get tech names and default interest rates
cfg = yaml.safe_load(TECH_YAML.read_text())
techs_cfg = cfg.get('techs', {})

# Build a list of (tech_name, interest_rate) from the YAML
tech_rates = {}
for name, params in techs_cfg.items():
    if isinstance(params, dict) and 'interest_rate' in params:
        tech_rates[name] = float(params['interest_rate'])

print(f'Found {len(tech_rates)} techs with interest rates:')
for t, r in sorted(tech_rates.items()):
    print(f'  {t}: {r:.4f}')

In [ ]:
# Load max capacities to get all locations and their land fraction
mc = pd.read_csv(MAX_CAPACITIES_CSV)

# Filter to cells with capacity
if 'max_capacity_mw' in mc.columns:
    mc = mc[mc['max_capacity_mw'] > 0].copy()

print(f'Total grid cells: {len(mc):,}')
print(f'  Pure ocean (land_pct == 0): {(mc.onshore_land_pct == 0).sum():,}')
print(f'  Coastal (0 < land_pct < 100): {((mc.onshore_land_pct > 0) & (mc.onshore_land_pct < 100)).sum():,}')
print(f'  Pure land (land_pct == 100): {(mc.onshore_land_pct == 100).sum():,}')

In [ ]:
# Build the spatial cost inputs DataFrame.
# One row per (location, tech) combination.

rows = []
tech_names = sorted(tech_rates.keys())

for _, loc in mc.iterrows():
    lat = float(loc['latitude'])
    lon = float(loc['longitude'])
    offshore_frac = float(loc['offshore_sea_pct']) / 100.0
    
    # TODO: vary offshore costs by distance-to-coast and fixed/floating wind depth bands
    # Scale multiplier by offshore fraction: 100% ocean -> full multiplier, 100% land -> 1.0
    # Scale multiplier by offshore fraction: 100% ocean → full multiplier, 100% land → 1.0
    build_mult = 1.0 + (OFFSHORE_BUILD_COST_MULTIPLIER - 1.0) * offshore_frac
    
    for tech in tech_names:
        rows.append({
            'lat': lat,
            'lon': lon,
            'tech': tech,
            'interest_rate': tech_rates[tech],
            'build_cost_multiplier': build_mult,
            'land_cost_usd_per_km2_year': DEFAULT_LAND_COST_USD_PER_KM2_YEAR,
            'water_cost_usd_per_m3': DEFAULT_WATER_COST_USD_PER_M3,
        })

sci_df = pd.DataFrame(rows)
print(f'Generated {len(sci_df):,} rows ({len(mc):,} locations x {len(tech_names)} techs)')

# Summarise offshore vs onshore
offshore_rows = sci_df[sci_df['build_cost_multiplier'] > 1.0]
print(f'Offshore rows (multiplier > 1): {len(offshore_rows):,} ({len(offshore_rows) // len(tech_names):,} locations)')
print(f'Onshore rows (multiplier == 1): {len(sci_df) - len(offshore_rows):,}')

In [ ]:
# Preview a few offshore entries
print('Sample offshore entries:')
display(sci_df[sci_df['build_cost_multiplier'] > 1.0].head(15))

print('\nSample onshore entries:')
display(sci_df[sci_df['build_cost_multiplier'] == 1.0].head(15))

In [ ]:
# Write to CSV
sci_df.to_csv(OUTPUT_CSV, index=False)
print(f'Saved spatial cost inputs to: {OUTPUT_CSV}')
print(f'File size: {OUTPUT_CSV.stat().st_size / 1024 / 1024:.1f} MB')